# Thí nghiệm Liên kết thực thể địa chỉ (Neural Entity Linking v1)
### Kiến trúc Shared Frozen ViDeBERTa-xsmall Encoder + Candidate Cache + Neural Linking Head

Notebook này được thiết kế để chạy **hoàn toàn tự động 100% (Run all)** trên Google Colab với T4 GPU.
- **Tác vụ**: Huấn luyện Đầu liên kết / Chấm ứng viên Tên - Mã hành chính 01/07/2025.
- **Nguyên tắc**: Encoder ViDeBERTa 70M đóng băng hoàn toàn (`requires_grad = False`). Chỉ cập nhật trọng số Neural Linking Head.
- **Tự động hóa**: Toàn bộ dữ liệu, bundle và checkpoint được nạp và lưu trữ trực tiếp trên Google Drive, không cần upload ZIP hay cấu hình đường dẫn thủ công.

**Hướng dẫn nhanh:**
1. Chọn **Runtime → Change runtime type → T4 GPU**.
2. Nhấn **Runtime → Run all** (hoặc `Ctrl + F9`) và cấp quyền truy cập Google Drive khi được yêu cầu.

In [ ]:
import sys, subprocess, torch
print('=== KIỂM TRA MÔI TRƯỜNG PHẦN CỨNG ===')
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA Khả dụng:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Thiết bị:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('[CẢNH BÁO] Không tìm thấy GPU. Hãy chọn Runtime -> Change runtime type -> T4 GPU để tăng tốc!')


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, shutil, zipfile, hashlib, subprocess

print('=== MOUNT GOOGLE DRIVE ===')
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/nlp_chuan_hoa_dia_chi')
EXP_DIR = DRIVE_BASE / 'linking_v1'
PROJECT_DIR = Path('/content/nlp_chuan_hoa_dia_chi')

print('Thư mục gốc Drive:', DRIVE_BASE)
print('Thư mục thí nghiệm:', EXP_DIR)
assert DRIVE_BASE.exists(), f'Không tìm thấy {DRIVE_BASE} trên Drive!'
assert EXP_DIR.exists(), f'Không tìm thấy {EXP_DIR} trên Drive!'


In [ ]:
print('=== GIẢI NÉN BUNDLE DỰ ÁN TỰ THÂN ===')
BUNDLE_ZIP = EXP_DIR / 'nlp_linking_bundle_v1.zip'
assert BUNDLE_ZIP.exists(), f'Lỗi: Không tìm thấy file bundle tại {BUNDLE_ZIP}'

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Đang giải nén {BUNDLE_ZIP.name} ({BUNDLE_ZIP.stat().st_size / 2**20:.2f} MB)...')
with zipfile.ZipFile(BUNDLE_ZIP, 'r') as z:
    z.extractall(PROJECT_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Giải nén thành công vào:', PROJECT_DIR)
print('Số tệp tin giải nén:', len(list(PROJECT_DIR.rglob('*'))))


In [ ]:
print('=== CÀI ĐẶT CÁC THƯ VIỆN PHỤ THUỘC ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'colab/linking_v1/requirements.txt'], check=True)
print('Cài đặt thư viện hoàn tất thành công!')


In [ ]:
print('=== CHẠY KIỂM THỬ ĐỘC LẬP (UNIT TESTS) ===')
subprocess.run([sys.executable, '-m', 'unittest', 'tests/test_lien_ket_v1.py'], check=True)
print('Tất cả unit tests đã PASS 100%!')


In [ ]:
print('=== THIẾT LẬP CHECKPOINT VÀ CACHE ===')
VIDEBERTA_MODEL_DIR = DRIVE_BASE / 'videberta_v1_01/model'
assert VIDEBERTA_MODEL_DIR.exists(), f'Không tìm thấy checkpoint ViDeBERTa tại {VIDEBERTA_MODEL_DIR}'

OUTPUT_CKPT_DIR = EXP_DIR / 'checkpoints'
CACHE_DIR = EXP_DIR / 'cache'
REPORT_DIR = EXP_DIR / 'reports'

OUTPUT_CKPT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('ViDeBERTa Checkpoint:', VIDEBERTA_MODEL_DIR)
print('Checkpoints Output:  ', OUTPUT_CKPT_DIR)
print('Candidate Cache Dir: ', CACHE_DIR)
print('Reports Output:      ', REPORT_DIR)


In [ ]:
print('=== HUẤN LUYỆN ĐẦU LIÊN KẾT THỰC THỂ (NEURAL LINKING HEAD) ===')
# ViDeBERTa encoder đóng băng 100%, chỉ linking head nhận gradient
# Chạy pilot đo throughput và in bảng ETA trước khi train full
train_cmd = [
    sys.executable, '-u', 'colab/linking_v1/train_linking.py',
    '--model_dir', str(VIDEBERTA_MODEL_DIR),
    '--data_path', 'data/train_dev/duyet_v2_01/train_cnn_ket_hop.jsonl',
    '--output_dir', str(OUTPUT_CKPT_DIR),
    '--cache_dir', str(CACHE_DIR),
    '--epochs', '5',
    '--batch_size', '32',
    '--lr', '0.001',
    '--pilot_steps', '20',
    '--device', 'auto',
    '--resume'
]
subprocess.run(train_cmd, check=True)


In [ ]:
print('=== ĐÁNH GIÁ CHẤT LƯỢNG VÀ ĐỘ TRỄ CPU ===')
eval_cmd = [
    sys.executable, '-u', 'colab/linking_v1/evaluate_linking.py',
    '--model_dir', str(VIDEBERTA_MODEL_DIR),
    '--head_path', str(OUTPUT_CKPT_DIR / 'best_linking_head.pt'),
    '--cache_dir', str(CACHE_DIR),
    '--data_path', 'data/train_dev/duyet_v2_01/train_cnn_ket_hop.jsonl',
    '--output_dir', str(REPORT_DIR),
    '--dev_goc_path', 'mo_hinh/thu_nghiem/char_v1_01/snapshot/dev.csv',
    '--dev_kho_path', 'mo_hinh/thu_nghiem/char_v1_01/snapshot/dev_kho.csv',
    '--device', 'cpu'
]
subprocess.run(eval_cmd, check=True)


In [ ]:
from IPython.display import Markdown, display
report_file = REPORT_DIR / 'BAO_CAO_DANH_GIA.md'
if report_file.exists():
    display(Markdown(report_file.read_text(encoding='utf-8')))
else:
    print('Không tìm thấy file báo cáo markdown tại:', report_file)


In [ ]:
print('=== ĐỒNG BỘ ARTIFACTS KẾT QUẢ VỀ GOOGLE DRIVE ===')
history_file = OUTPUT_CKPT_DIR / 'training_history.json'
if history_file.exists():
    print('Lịch sử huấn luyện:', history_file)
best_head = OUTPUT_CKPT_DIR / 'best_linking_head.pt'
if best_head.exists():
    print(f'Mô hình Linking Head tốt nhất: {best_head} ({best_head.stat().st_size / 1024:.1f} KB)')
print('Toàn bộ kết quả thí nghiệm đã được lưu an toàn tại:', EXP_DIR)
print('HOÀN TẤT THÀNH CÔNG 100%!')
